# Génération de population — eqasim

Ce notebook orchestre la création d'une population synthétique pour la simulation GAMA / LLM-agents.

## Ce que fait ce notebook

1. **Génère** une population via le service eqasim (port 8003) — appel HTTP bloquant, peut durer plusieurs minutes
2. **Valide et corrige** les séquences d'activités (chevauchements, durées nulles, ordre chronologique)
3. **Enrichit** chaque localisation avec un flag `public_transport` (distance au stop GTFS ≤ 1 500 m)
4. **Calcule** les itinéraires manquants (foot / bicycle / car) via OSMnx en parallèle sur plusieurs workers
5. **Initialise** les horaires de départ (`scheduled_start_time`) et ajuste les gaps de trajet selon les priorités d'activités
6. **Sauvegarde** atomiquement les fichiers JSON complétés dans `data/eqasim_output/`

## Prérequis

| Service | Commande de démarrage | Port |
|---|---|---|
| eqasim | `docker compose up eqasim` | 8003 |
| OSMnx cache | généré automatiquement à la 1ʳᵉ exécution (~1–2 min) | — |

Dépendances Python : `numpy`, `pandas`, `tqdm`, `osmnx`

## Données en entrée

- `data/gtfs/tisseo_gtfs/stops.txt` — arrêts Tisséo pour l'enrichissement du flag PT
- `data/osmnx_cache/` — graphes routiers (créés automatiquement si absents)
- `llm-agents/` — modules `trip_helper` et `route_worker` utilisés pour le calcul d'itinéraires

## Fichiers produits

- `data/eqasim_output/toulouse_population_<N>.json` — population enrichie, prête pour la simulation

## Durée estimée

| Étape | Durée indicative |
|---|---|
| Génération eqasim (100 agents) | ~10–30 s |
| Calcul des itinéraires OSMnx (1 000 paires, 12 workers) | ~6 min |

## Enchaînement recommandé

1. S'assurer que le service eqasim est démarré
2. Exécuter toutes les cellules dans l'ordre (`Run All`)
3. Lancer ensuite `statistics_population.ipynb` pour visualiser les statistiques

In [7]:
# ── Paramètres ────────────────────────────────────────────────────────────────
POPULATION_SIZE       = 100   # Nombre d'agents souhaités
GENERATE_PERSONALITY  = False  # True → génère les Big Five (lent)
FORCE_REGENERATE      = False  # True → ignore le cache eqasim

# BBox optionnelle [min_lon, min_lat, max_lon, max_lat] en WGS84
# None → département 31 complet
BBOX = None
# BBOX = [1.35, 43.55, 1.50, 43.65]  # exemple : centre de Toulouse

EQASIM_URL = "http://localhost:8003"


In [8]:
import json
import re
import time
from pathlib import Path

REPO_ROOT = Path("../../../").resolve()
POP_DIR   = REPO_ROOT / "data" / "eqasim_output"
POP_DIR.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"POP_DIR   : {POP_DIR}")
print(f"population_size={POPULATION_SIZE}  personality={GENERATE_PERSONALITY}  force={FORCE_REGENERATE}  bbox={BBOX}")

REPO_ROOT : /Users/yvesb/Documents/llm-agents-gama
POP_DIR   : /Users/yvesb/Documents/llm-agents-gama/data/eqasim_output
population_size=100  personality=False  force=False  bbox=None


In [9]:
# ── Vérification santé du service eqasim ─────────────────────────────────────
import urllib


def check_health(url: str, retries: int = 3, delay: float = 2.0) -> bool:
    for i in range(retries):
        try:
            with urllib.request.urlopen(f"{url}/health", timeout=5) as r:
                return r.status == 200
        except Exception as e:
            print(f"  tentative {i+1}/{retries} : {e}")
            if i < retries - 1:
                time.sleep(delay)
    return False

if check_health(EQASIM_URL):
    print(f"Service eqasim OK → {EQASIM_URL}")
else:
    raise RuntimeError(
        f"Service eqasim inaccessible sur {EQASIM_URL}.\n"
        "Démarrez le service avec : docker compose up eqasim"
    )

Service eqasim OK → http://localhost:8003


In [10]:
# ── Génération de la population ───────────────────────────────────────────────
expected = POP_DIR / f"toulouse_population_{POPULATION_SIZE}.json"

if not FORCE_REGENERATE and expected.exists():
    print(f"Fichier {expected.name} déjà présent — génération ignorée.")
    print("(Mettre FORCE_REGENERATE = True pour forcer la régénération.)")
else:
    payload = {
        "population_size":      POPULATION_SIZE,
        "generate_personality": GENERATE_PERSONALITY,
        "force":                FORCE_REGENERATE,
    }
    if BBOX is not None:
        payload["bbox"] = BBOX

    body = json.dumps(payload).encode()
    req  = urllib.request.Request(
        f"{EQASIM_URL}/generate",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    print(f"POST {EQASIM_URL}/generate  payload={payload}")
    print("En attente de la réponse (bloquant — peut prendre plusieurs minutes)…")
    t0 = time.monotonic()

    try:
        with urllib.request.urlopen(req, timeout=3600) as resp:
            result = json.loads(resp.read())
    except urllib.error.HTTPError as e:
        result = json.loads(e.read())
        print(f"HTTP {e.code} : {result}")
        raise

    elapsed = time.monotonic() - t0
    print(f"Réponse reçue en {elapsed:.1f}s : {result}")

    if result.get("status") != "ok":
        raise RuntimeError(f"Echec eqasim : {result}")


POST http://localhost:8003/generate  payload={'population_size': 100, 'generate_personality': False, 'force': False}
En attente de la réponse (bloquant — peut prendre plusieurs minutes)…
Réponse reçue en 9.3s : {'status': 'ok', 'file': '/eqasim-output/toulouse_population_100.json'}


In [11]:
# ── Vérification du fichier de sortie ────────────────────────────────────────
pattern = re.compile(r"^toulouse_population_(\d+)\.json$")
pop_files = sorted(
    POP_DIR.glob("toulouse_population_*.json"),
    key=lambda p: int(pattern.match(p.name).group(1)),
    reverse=True,
)

expected = POP_DIR / f"toulouse_population_{POPULATION_SIZE}.json"

print(f"Fichiers présents dans {POP_DIR.relative_to(REPO_ROOT)}:")
for p in pop_files:
    size_mb = p.stat().st_size / 1_048_576
    tag = " ← généré" if p == expected else ""
    print(f"  {p.name}  ({size_mb:.1f} Mo){tag}")

if not expected.exists():
    if pop_files:
        print(f"\nAvertissement : {expected.name} absent, fichier le plus récent : {pop_files[0].name}")
    else:
        raise FileNotFoundError(f"Aucun fichier de population dans {POP_DIR}")
else:
    print(f"\nFichier cible prêt : {expected}")

Fichiers présents dans data/eqasim_output:
  toulouse_population_100.json  (0.4 Mo) ← généré

Fichier cible prêt : /Users/yvesb/Documents/llm-agents-gama/data/eqasim_output/toulouse_population_100.json


In [12]:
# ── Aperçu rapide de la population générée ───────────────────────────────────
target = expected if expected.exists() else pop_files[0]

with open(target, encoding="utf-8") as f:
    data = json.load(f)

n_people = len(data)
n_acts   = sum(len(e.get("identity", {}).get("activities", [])) for e in data)
ages     = [e["identity"]["traits_json"].get("age") for e in data
            if "traits_json" in e.get("identity", {})]
ages     = [a for a in ages if a is not None]

print(f"Fichier         : {target.name}")
print(f"Personnes       : {n_people}")
print(f"Activités total : {n_acts}  (moy. {n_acts/n_people:.2f}/personne)")
if ages:
    print(f"Âge moy.        : {sum(ages)/len(ages):.1f} ans  [{min(ages)}, {max(ages)}]")

purposes: dict[str, int] = {}
for e in data:
    for a in e.get("identity", {}).get("activities", []):
        p = a.get("purpose", "?")
        purposes[p] = purposes.get(p, 0) + 1
print("\nRépartition des activités :")
for purpose, cnt in sorted(purposes.items(), key=lambda x: -x[1]):
    print(f"  {purpose:<12} : {cnt:5d}  ({cnt/n_acts*100:.1f}%)")

print("\n→ Lancer statistics_population.ipynb pour visualiser les statistiques.")

Fichier         : toulouse_population_100.json
Personnes       : 108
Activités total : 507  (moy. 4.69/personne)
Âge moy.        : 39.3 ans  [5, 88]

Répartition des activités :
  home         :   279  (55.0%)
  work         :    84  (16.6%)
  leisure      :    52  (10.3%)
  other        :    46  (9.1%)
  shop         :    37  (7.3%)
  education    :     9  (1.8%)

→ Lancer statistics_population.ipynb pour visualiser les statistiques.


## Post-processing — Vérification & Complétion

Les sections suivantes sont indépendantes de la génération eqasim et peuvent être relancées
sans régénérer la population (ex. après un changement de données GTFS ou de graphe OSMnx).

Elles opèrent sur **tous les fichiers JSON** présents dans `data/eqasim_output/` et maintiennent
les données en mémoire (`all_data`) ; la sauvegarde sur disque n'a lieu qu'en **Section 6**.

| Section | Rôle |
|---|---|
| 1 | Chargement et inventaire de complétude |
| 1b | Validation et correction des séquences d'activités |
| 2 | Enrichissement du flag `public_transport` (GTFS) |
| 3 | Détection des paires d'activités sans itinéraire |
| 4 | Calcul des itinéraires manquants (OSMnx, parallèle) |
| 5 | Injection des routes calculées dans le JSON |
| 5b | Initialisation des `scheduled_start_time` |
| 5b bis | Ajustement des gaps de trajet (priorités d'activités) |
| 5c | Vérification finale de faisabilité des déplacements |
| 6 | Sauvegarde atomique sur disque |

In [ ]:
import os
import sys
import hashlib
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# ── Chemins additionnels ──────────────────────────────────────────────────────
GTFS_STOPS     = REPO_ROOT / "data" / "gtfs" / "tisseo_gtfs" / "stops.txt"
OSMNX_CACHE    = REPO_ROOT / "data" / "osmnx_cache"
SCRIPTS_POP    = REPO_ROOT / "scripts" / "data" / "population"
LLMAGENTS_PATH = str(REPO_ROOT / "llm-agents")

# Clé de cache OSMnx : identifie le graphe routier chargé pour Toulouse (rayon 30 km)
CACHE_KEY = hashlib.md5(b"Toulouse, France_30000").hexdigest()[:12]
MAX_WORKERS   = 12       # workers parallèles pour le calcul d'itinéraires (ProcessPoolExecutor)
TRIP_MODES    = ["foot", "bicycle", "car"]
MAX_PT_DIST_M = 1500.0   # seuil (mètres) en dessous duquel une localisation est considérée desservie en TC

if LLMAGENTS_PATH not in sys.path:
    sys.path.insert(0, LLMAGENTS_PATH)
SCRIPTS_POP_STR = str(SCRIPTS_POP)
if SCRIPTS_POP_STR not in sys.path:
    sys.path.insert(0, SCRIPTS_POP_STR)

print(f"GTFS_STOPS:  {GTFS_STOPS}")
print(f"OSMNX_CACHE: {OSMNX_CACHE}")
print(f"OSMnx cache key: {CACHE_KEY}")

## Section 1 — Chargement & Inventaire

Charge tous les fichiers `toulouse_population_*.json` présents dans `POP_DIR`,
puis calcule pour chacun un résumé de complétude :
- nombre de personnes et de paires d'activités consécutives (y compris la paire cyclique last→first)
- nombre de localisations sans flag `public_transport`
- nombre de paires sans itinéraire (`transfert_from_previous_location` absent ou incomplet)

In [ ]:
def load_file(path: Path) -> list:
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def _activity_index_pairs(activities: list, has_car: bool) -> list[tuple]:
    """Return all (prev_i, curr_i, modes) tuples to check, including the cyclic last→first."""
    n = len(activities)
    modes = TRIP_MODES if has_car else ["foot", "bicycle"]
    pairs = [(i - 1, i, modes) for i in range(1, n)]
    if n >= 2:
        pairs.append((n - 1, 0, modes))  # paire cyclique : retour au domicile (last → first)
    return pairs

def check_enrichment(data: list) -> dict:
    """
    Retourne des compteurs de complétude pour une liste de personnes :
      - total_people     : nombre de personnes
      - total_pairs      : paires d'activités consécutives, y compris la paire cyclique last→first
      - missing_pt       : localisations sans champ public_transport
      - missing_routes   : paires sans transfert_from_previous_location
      - missing_any_mode : paires présentes mais incomplètes (au moins un mode manquant)
    """
    total_people   = len(data)
    total_pairs    = 0
    missing_pt     = 0
    missing_routes = 0
    missing_any    = 0

    for entry in data:
        identity   = entry.get("identity", {})
        activities = identity.get("activities", [])
        traits     = identity.get("traits_json", {})
        has_car    = traits.get("number_of_cars", 0) > 0
        expected_modes = set(TRIP_MODES) if has_car else {"foot", "bicycle"}

        # Vérification du domicile
        home = identity.get("home")
        if home and "public_transport" not in home:
            missing_pt += 1

        # Vérification des lieux d'activité
        for act in activities:
            loc = act.get("location")
            if loc and "public_transport" not in loc:
                missing_pt += 1

        # Vérification des itinéraires entre activités consécutives
        for prev_i, curr_i, _ in _activity_index_pairs(activities, has_car):
            total_pairs += 1
            routes = activities[curr_i].get("transfert_from_previous_location")
            if not routes:
                missing_routes += 1
            else:
                if not expected_modes.issubset(set(routes.keys())):
                    missing_any += 1

    return {
        "total_people":   total_people,
        "total_pairs":    total_pairs,
        "missing_pt":     missing_pt,
        "missing_routes": missing_routes,
        "missing_any":    missing_any,
    }

# Chargement de tous les fichiers de population présents dans POP_DIR
pop_files = sorted(POP_DIR.glob("*.json"))
print(f"Found {len(pop_files)} population files:")
for p in pop_files:
    print(f"  {p.name}")

all_data: dict[str, list] = {}
status_rows = []

for p in pop_files:
    data = load_file(p)
    all_data[p.name] = data
    stats = check_enrichment(data)
    stats["file"] = p.name
    status_rows.append(stats)

status_df = pd.DataFrame(status_rows).set_index("file")
print("\nInventory:")
display(status_df)

## Section 1b — Activity Validation & Correction

Rules enforced on every person's activity list (circular-day model):
1. **No overlap** : `start_time[k+1] >= end_time[k]` — the gap between two consecutive activities is ≥ 0 and represents the travel time from ENTD data. A positive gap is valid and preserved. A negative gap (overlap) is fixed by setting `start_time[k+1] = end_time[k]` (option B: gap → 0, no ENTD travel time for this pair).
2. **Non-zero duration** : `start_time != end_time` for every activity (`start > end` is valid for the midnight-crossing activity)
3. **Chronological order** : `start_time[i] > start_time[i-1]` for `i = 1 … n-1`, skipped when the predecessor crosses midnight

`fix_activities` also performs the **circular merge**: if `activities[0].purpose == activities[-1].purpose` the two activities are folded into one (the last is dropped, the first inherits its `start_time`).

In [15]:
import copy
import uuid as _uuid

MAX_DAY_SECONDS = 86_400.0  # 24 h in seconds


def _make_home_activity(home_loc: dict, start_time: float = 0.0, end_time: float = 86400.0) -> dict:
    return {
        "id": str(_uuid.uuid4()),
        "scheduled_start_time": None,
        "start_time": start_time,
        "end_time": end_time,
        "purpose": "home",
        "location": home_loc,
    }


# ── Check ─────────────────────────────────────────────────────────────────────

def check_activities(person: dict) -> list[str]:
    acts = person.get("identity", {}).get("activities", [])
    issues: list[str] = []

    if not acts:
        return ["No activities"]

    n = len(acts)

    # R2 — all times must be in [0, 86400]
    for i, act in enumerate(acts):
        for key in ("start_time", "end_time"):
            t = act.get(key)
            if t is not None and t > MAX_DAY_SECONDS:
                issues.append(
                    f"[R2] Activity[{i}] '{act.get('purpose', '?')}': "
                    f"{key}={t} exceeds 86400"
                )

    # R3 — non-zero duration: start == end is always invalid
    #       start > end is valid (activity crosses midnight)
    for i, act in enumerate(acts):
        st = act.get("start_time")
        et = act.get("end_time")
        p  = act.get("purpose", "?")
        if st is not None and et is not None and st == et:
            issues.append(f"[R3] Activity[{i}] '{p}': zero duration (start == end == {st})")

    if n == 1:
        return issues

    # R4 — strict chronological order for i = 1 ... n-1
    #       skipped when the predecessor crosses midnight (start > end)
    for i in range(1, n):
        prev = acts[i - 1]
        prev_st = prev.get("start_time")
        prev_et = prev.get("end_time")
        curr_st = acts[i].get("start_time")
        if prev_st is None or prev_et is None or curr_st is None:
            continue
        if prev_st > prev_et:
            continue  # predecessor crosses midnight
        if curr_st <= prev_st:
            issues.append(
                f"[R4] Activity[{i}] '{acts[i].get('purpose')}': "
                f"start_time={curr_st} <= previous {prev_st}"
            )

    # R1 — no overlap: each activity must start at or after the previous ends
    #       a positive gap (travel time from ENTD) is valid and preserved
    for i in range(1, n):
        curr_st = acts[i].get("start_time")
        prev_et = acts[(i - 1)].get("end_time")
        if curr_st is not None and prev_et is not None and curr_st < prev_et:
            issues.append(
                f"[R1] Activity[{i}] '{acts[i].get('purpose')}': "
                f"overlap: start_time={curr_st} < prev end_time={prev_et}"
            )

    return issues


# ── Fix ───────────────────────────────────────────────────────────────────────

def fix_activities(person: dict) -> tuple[dict, list[str]]:
    """
    Fix order:
      0. Normalize times to [0, 86400) with modulo 24h
      1. Sort by start_time
      2. Fix zero-duration or inverted activities
      3. Circular merge: fold first+last if same purpose
      4. Fix overlaps only: if start_time[k+1] < end_time[k], set start_time[k+1] = end_time[k].
         Positive gaps (travel time from ENTD) are preserved as-is.
    """
    person   = copy.deepcopy(person)
    identity = person.get("identity", {})
    acts: list[dict] = identity.get("activities", [])
    home_loc: dict   = identity.get("home", {})
    fixes: list[str] = []

    if not acts:
        acts = [_make_home_activity(home_loc)]
        fixes.append("Created single home activity (list was empty)")
        identity["activities"] = acts
        return person, fixes

    # 0 — normalize times to [0, 86400) using modulo 24h
    for i, act in enumerate(acts):
        for key in ("start_time", "end_time"):
            t = act.get(key)
            if t is not None and t > MAX_DAY_SECONDS:
                new_t = t % MAX_DAY_SECONDS
                fixes.append(
                    f"Activity[{i}] '{act.get('purpose')}': "
                    f"{key} {t:.0f} -> {new_t:.0f} (modulo 24h)"
                )
                act[key] = new_t

    # 1 — sort by start_time
    original_order = [a.get("start_time") for a in acts]
    acts.sort(key=lambda a: a.get("start_time") or 0.0)
    if [a.get("start_time") for a in acts] != original_order:
        fixes.append("Reordered activities by start_time")

    # 2 — fix zero-duration or inverted activities
    for i, act in enumerate(acts):
        st = act.get("start_time")
        et = act.get("end_time")
        if st is None or et is None:
            continue
        if st > et:
            act["start_time"], act["end_time"] = et, st
            fixes.append(
                f"Activity[{i}] '{act.get('purpose')}': "
                f"swapped start_time/end_time ({st:.0f} <-> {et:.0f})"
            )
        elif st == et:
            act["end_time"] = st + 600.0
            fixes.append(
                f"Activity[{i}] '{act.get('purpose')}': "
                f"zero duration, added 10 min gap -> end_time={act['end_time']:.0f}"
            )

    # 3 — circular merge: fold first+last if same purpose
    if len(acts) >= 2 and acts[0].get("purpose") == acts[-1].get("purpose"):
        last = acts.pop()
        acts[0]["start_time"] = last["start_time"]
        fixes.append(
            f"Circular merge: folded first+last '{acts[0].get('purpose')}' "
            f"(start_time {last['start_time']:.0f} -> {acts[0]['end_time']:.0f}, crosses midnight)"
        )

    # 4 — fix overlaps: if start_time[k+1] < end_time[k], collapse gap to 0 (option B).
    #     Positive gaps representing ENTD travel time are preserved untouched.
    n = len(acts)
    if n >= 2:
        for i in range(1, n):
            prev_et = acts[i - 1].get("end_time")
            curr_st = acts[i].get("start_time")
            if prev_et is not None and curr_st is not None and curr_st < prev_et:
                fixes.append(
                    f"Activity[{i}] '{acts[i].get('purpose')}': "
                    f"overlap fixed: start_time {curr_st:.0f} -> {prev_et:.0f}"
                )
                acts[i]["start_time"] = prev_et

    identity["activities"] = acts
    return person, fixes


print("check_activities() and fix_activities() ready.")

check_activities() and fix_activities() ready.


In [16]:
# ── Report violations across all loaded files (read-only) ─────────────────────
rule_labels = {
    "R1": "Cyclic continuity violated (gap or overlap)",
    "R3": "Zero-duration activity (start == end)",
    "R4": "Out-of-order activities",
}
violation_counts = {r: 0 for r in rule_labels}
total_invalid = 0

for fname, data in all_data.items():
    file_invalid = 0
    for person in data:
        issues = check_activities(person)
        if issues:
            file_invalid += 1
            total_invalid += 1
            for issue in issues:
                for rule in rule_labels:
                    if f"[{rule}]" in issue:
                        violation_counts[rule] += 1
    print(f"{fname}: {file_invalid}/{len(data)} persons with violations")

print()
print("Violation breakdown (all files combined):")
for rule, count in violation_counts.items():
    marker = "  " if count == 0 else "⚠ "
    print(f"  {marker}[{rule}] {rule_labels[rule]}: {count}")
print(f"\nTotal persons with at least one violation: {total_invalid}")


toulouse_population_100.json: 0/108 persons with violations

Violation breakdown (all files combined):
    [R1] Cyclic continuity violated (gap or overlap): 0
    [R3] Zero-duration activity (start == end): 0
    [R4] Out-of-order activities: 0

Total persons with at least one violation: 0


In [17]:
# ── Apply fixes in-memory (files will be saved by Section 6) ──────────────────
total_persons_fixed = 0
total_corrections   = 0

for fname, data in all_data.items():
    persons_fixed = 0
    corrections   = 0
    fixed_data    = []

    for person in data:
        if check_activities(person):
            fixed_person, applied_fixes = fix_activities(person)
            # Verify the fix was complete
            remaining = check_activities(fixed_person)
            if remaining:
                pid = person.get("person_id", "?")
                print(f"  !! {fname} person {pid}: still invalid after fix: {remaining}")
            fixed_data.append(fixed_person)
            persons_fixed += 1
            corrections   += len(applied_fixes)
        else:
            fixed_data.append(person)

    # Update in-memory dataset (Section 6 will persist to disk)
    all_data[fname] = fixed_data
    total_persons_fixed += persons_fixed
    total_corrections   += corrections
    print(f"{fname}: {persons_fixed} persons fixed ({corrections} corrections)")

print(f"\nTotal: {total_persons_fixed} persons fixed, {total_corrections} individual corrections")
print()

# ── Confirm 0 violations remain ───────────────────────────────────────────────
remaining_invalid = sum(
    1 for data in all_data.values()
      for person in data
      if check_activities(person)
)
if remaining_invalid == 0:
    print("✓ All activity lists are now valid.")
else:
    print(f"⚠ {remaining_invalid} persons still invalid — check output above.")

toulouse_population_100.json: 0 persons fixed (0 corrections)

Total: 0 persons fixed, 0 individual corrections

✓ All activity lists are now valid.


In [18]:
# ── Circular merge: unconditional pass for ALL persons ─────────────────────────
# fix_activities only runs on invalid persons, so valid persons with both a
# morning home and an evening home never get the fold. We apply it here to
# everyone: if first and last activities share the same purpose, pop the last
# and set first.start_time = last.start_time (midnight-crossing activity).
total_merged = 0
for fname, data in all_data.items():
    n_merged = 0
    for person in data:
        acts = person.get('identity', {}).get('activities', [])
        if len(acts) >= 2 and acts[0].get('purpose') == acts[-1].get('purpose'):
            last = acts.pop()
            acts[0]['start_time'] = last['start_time']
            acts[0]['scheduled_start_time'] = None  # will be recomputed
            n_merged += 1
    total_merged += n_merged
    print(f'{fname}: {n_merged} circular merges applied')
print(f'\nTotal merged: {total_merged}')


toulouse_population_100.json: 100 circular merges applied

Total merged: 100


## Section 2 — Enrichissement des flags `public_transport`

Pour chaque personne, marque le domicile et chaque lieu d'activité avec
`"public_transport": true/false` selon que la localisation se trouve à moins de
`MAX_PT_DIST_M` mètres (1 500 m par défaut) d'un arrêt Tisséo du fichier GTFS.

Le calcul de distance utilise une approximation euclidéenne vectorisée (≈ ±0,1 % sur la zone de Toulouse).

In [19]:
# Load GTFS stops
stops_df = pd.read_csv(GTFS_STOPS, usecols=["stop_lat", "stop_lon"])
stop_lats = stops_df["stop_lat"].values
stop_lons = stops_df["stop_lon"].values
print(f"Loaded {len(stops_df)} GTFS stops")

def nearest_stop_dist_m(lat: float, lon: float) -> float:
    """Vectorised Euclidean approximation for nearest stop distance in metres."""
    dlat = (stop_lats - lat) * 111_320.0
    dlon = (stop_lons - lon) * 111_320.0 * np.cos(np.radians(lat))
    return float(np.hypot(dlat, dlon).min())

def enrich_public_transport(data: list) -> int:
    """Add / overwrite public_transport field on every location. Returns count of locations updated."""
    updated = 0
    for entry in data:
        identity = entry.get("identity", {})
        home = identity.get("home")
        if home and home.get("lat") is not None:
            pt = nearest_stop_dist_m(home["lat"], home["lon"]) <= MAX_PT_DIST_M
            if home.get("public_transport") != pt:
                home["public_transport"] = pt
                updated += 1

        for act in identity.get("activities", []):
            loc = act.get("location")
            if loc and loc.get("lat") is not None:
                pt = nearest_stop_dist_m(loc["lat"], loc["lon"]) <= MAX_PT_DIST_M
                if loc.get("public_transport") != pt:
                    loc["public_transport"] = pt
                    updated += 1
    return updated

for fname, data in all_data.items():
    n = enrich_public_transport(data)
    print(f"{fname}: {n} location(s) updated with public_transport flag")

Loaded 5661 GTFS stops
toulouse_population_100.json: 515 location(s) updated with public_transport flag


## Section 3 — Collecte des paires d'activités sans itinéraire

Parcourt toutes les personnes et identifie chaque paire (origine, destination, mode) pour laquelle
`transfert_from_previous_location` est absent ou incomplet. Les paires identiques entre plusieurs
personnes sont dédupliquées dans `global_missing` pour éviter les calculs redondants en Section 4.

La paire cyclique (dernière activité → première activité, i.e. le retour au domicile en fin de journée)
est systématiquement incluse.

In [20]:
def collect_pairs_for_file(data: list) -> set:
    """
    For each person, iterate ALL consecutive activity pairs (including "other")
    and the cyclic pair (last → first).
    Return a set of (lat1, lon1, lat2, lon2, mode) tuples where the route is missing.
    """
    pairs = set()
    for entry in data:
        identity   = entry.get("identity", {})
        activities = identity.get("activities", [])
        traits     = entry["identity"].get("traits_json", {})
        has_car    = traits.get("number_of_cars", 0) > 0

        for prev_i, curr_i, modes in _activity_index_pairs(activities, has_car):
            prev_act = activities[prev_i]
            curr_act = activities[curr_i]
            prev_loc = prev_act.get("location")
            curr_loc = curr_act.get("location")
            if not prev_loc or not curr_loc:
                continue
            if prev_loc.get("lat") is None or curr_loc.get("lat") is None:
                continue
            routes = curr_act.get("transfert_from_previous_location") or {}
            for mode in modes:
                if mode not in routes:
                    pairs.add((
                        round(prev_loc["lat"], 7), round(prev_loc["lon"], 7),
                        round(curr_loc["lat"], 7), round(curr_loc["lon"], 7),
                        mode,
                    ))
    return pairs

global_missing: set = set()
per_file_missing: dict[str, set] = {}

for fname, data in all_data.items():
    pairs = collect_pairs_for_file(data)
    per_file_missing[fname] = pairs
    global_missing |= pairs
    print(f"{fname}: {len(pairs)} missing (lat1,lon1,lat2,lon2,mode) pairs")

print(f"\nGlobal unique pairs to compute across all files: {len(global_missing)}")

toulouse_population_100.json: 1007 missing (lat1,lon1,lat2,lon2,mode) pairs

Global unique pairs to compute across all files: 1007


## Section 4 — Calcul des itinéraires manquants (OSMnx, parallèle)

Lance un `ProcessPoolExecutor` avec `MAX_WORKERS` workers pour calculer en parallèle tous
les itinéraires identifiés en Section 3. Chaque worker charge le graphe OSMnx de Toulouse
(depuis le cache disque) à son initialisation, puis calcule les routes par lots (`chunksize=20`).

L'heure de référence pour les calculs de congestion est fixée à **8h30** (heure de pointe matinale).
Les routes inaccessibles (ex. îles, zones hors graphe) sont stockées comme `None` dans le cache.

In [21]:
from route_worker import init_worker, compute_route_worker

route_cache: dict[tuple, dict | None] = {}

if not global_missing:
    print("All routes already present — nothing to compute.")
else:
    # Use 8:30 AM (morning commute) as default congestion hour
    DEFAULT_HOUR = 8.5

    tasks = [(lat1, lon1, lat2, lon2, mode, DEFAULT_HOUR)
             for (lat1, lon1, lat2, lon2, mode) in global_missing]

    print(f"Computing {len(tasks)} routes with {MAX_WORKERS} workers…")
    t0 = time.monotonic()

    import multiprocessing
    ctx = multiprocessing.get_context("spawn")
    with ProcessPoolExecutor(
        max_workers=MAX_WORKERS,
        mp_context=ctx,
        initializer=init_worker,
        initargs=(LLMAGENTS_PATH, str(OSMNX_CACHE), CACHE_KEY),
    ) as pool:
        results = list(tqdm(
            pool.map(compute_route_worker, tasks, chunksize=20),
            total=len(tasks),
            desc="routing",
        ))

    elapsed = time.monotonic() - t0
    print(f"Done in {elapsed:.1f}s  ({elapsed/len(tasks)*1000:.1f} ms/route avg)")

    ok_count   = 0
    null_count = 0
    for args, result in results:
        lat1, lon1, lat2, lon2, mode, _ = args
        key = (lat1, lon1, lat2, lon2, mode)
        route_cache[key] = result
        if result is None:
            null_count += 1
        else:
            ok_count += 1

    print(f"Routes computed: {ok_count} OK, {null_count} unreachable (None)")

Computing 1007 routes with 12 workers…


routing:   0%|          | 0/1007 [00:00<?, ?it/s]

[worker pid=44192] graphs loaded in 51.2s
[worker pid=44193] graphs loaded in 55.7s
[worker pid=44194] graphs loaded in 59.1s
[worker pid=44188] graphs loaded in 59.2s


2026-05-27 13:27:27.949 | INFO     | trip_helper.osmnx_direct:<module>:62 - OSMnx HTTP concurrency limit: 3
2026-05-27 13:27:30.502 | INFO     | trip_helper.osmnx_direct:<module>:62 - OSMnx HTTP concurrency limit: 3


[worker pid=44185] graphs loaded in 68.4s


...

[worker pid=44191] graphs loaded in 68.9s


2026-05-27 13:27:35.366 | INFO     | trip_helper.osmnx_direct:<module>:62 - OSMnx HTTP concurrency limit: 3
2026-05-27 13:27:43.373 | INFO     | trip_helper.osmnx_direct:<module>:62 - OSMnx HTTP concurrency limit: 3
2026-05-27 13:27:43.458 | INFO     | trip_helper.osmnx_direct:<module>:62 - OSMnx HTTP concurrency limit: 3


[worker pid=44195] graphs loaded in 79.9s
[worker pid=44196] graphs loaded in 79.9s
[worker pid=44190] graphs loaded in 80.2s
[worker pid=44189] graphs loaded in 80.7s
[worker pid=44186] graphs loaded in 80.9s
[worker pid=44187] graphs loaded in 81.2s


2026-05-27 13:27:52.723 | INFO     | trip_helper.osmnx_direct:<module>:62 - OSMnx HTTP concurrency limit: 3
2026-05-27 13:27:53.017 | INFO     | trip_helper.osmnx_direct:<module>:62 - OSMnx HTTP concurrency limit: 3
2026-05-27 13:27:53.127 | INFO     | trip_helper.osmnx_direct:<module>:62 - OSMnx HTTP concurrency limit: 3
2026-05-27 13:27:53.456 | INFO     | trip_helper.osmnx_direct:<module>:62 - OSMnx HTTP concurrency limit: 3
2026-05-27 13:27:55.213 | INFO     | trip_helper.osmnx_direct:<module>:62 - OSMnx HTTP concurrency limit: 3
2026-05-27 13:27:55.648 | INFO     | trip_helper.osmnx_direct:<module>:62 - OSMnx HTTP concurrency limit: 3


Done in 375.6s  (373.0 ms/route avg)
Routes computed: 1002 OK, 5 unreachable (None)


## Section 5 — Injection des routes dans le JSON

Parcourt à nouveau toutes les paires d'activités et écrit dans `transfert_from_previous_location`
les routes calculées en Section 4. Seuls les modes manquants sont remplis ; les routes déjà
présentes dans le fichier ne sont pas écrasées.

La paire cyclique (last → first) est traitée comme les autres paires.

In [22]:
def apply_routes(data: list, cache: dict) -> int:
    """
    Write computed routes into each activity's transfert_from_previous_location,
    including the cyclic last→first pair (act[0]).
    Only fills in missing modes — existing routes are left untouched.
    Returns the number of (activity, mode) entries written.
    """
    written = 0
    for entry in data:
        identity   = entry.get("identity", {})
        all_acts   = identity.get("activities", [])
        traits     = identity.get("traits_json", {})
        has_car    = traits.get("number_of_cars", 0) > 0

        act_by_id = {a["id"]: a for a in all_acts}

        for prev_i, curr_i, modes in _activity_index_pairs(all_acts, has_car):
            prev_act = all_acts[prev_i]
            curr     = all_acts[curr_i]
            prev_loc = prev_act.get("location")
            curr_loc = curr.get("location")
            if not prev_loc or not curr_loc:
                continue
            if prev_loc.get("lat") is None or curr_loc.get("lat") is None:
                continue

            orig_act = act_by_id.get(curr["id"], curr)
            if not orig_act.get("transfert_from_previous_location"):
                orig_act["transfert_from_previous_location"] = {}

            for mode in modes:
                key = (
                    round(prev_loc["lat"], 7), round(prev_loc["lon"], 7),
                    round(curr_loc["lat"], 7), round(curr_loc["lon"], 7),
                    mode,
                )
                if key in cache and mode not in orig_act["transfert_from_previous_location"]:
                    orig_act["transfert_from_previous_location"][mode] = cache[key]
                    written += 1
    return written

for fname, data in all_data.items():
    written = apply_routes(data, route_cache)
    print(f"{fname}: {written} route entries written")

toulouse_population_100.json: 1147 route entries written


## Section 5b — Schedule Initialization (scheduled_start_time)

`scheduled_start_time[k]` represents the **departure time** from the previous activity toward activity k.
It is set to `end_time[k-1]` — the moment the person leaves activity k-1.

In the cyclic model: `scheduled_start_time[0] = end_time[last] % 86400`.

This replaces the former iterative OSMnx-propagation approach. The ENTD gap
(`start_time[k] - end_time[k-1]`) already encodes the observed travel time; the
OSMnx feasibility check is done separately in Section 5c.

In [23]:

def _min_trip_s(curr_act: dict) -> int:
    """Minimum travel time: best available mode, or crow-flies fallback."""
    routes = curr_act.get("transfert_from_previous_location") or {}
    durations = [
        v["duration_s"]
        for v in routes.values()
        if v and v.get("duration_s") is not None
    ]
    if durations:
        return min(durations)
    print(f"  !! Missing locations for min_trip_s fallback {curr_act}")
    return 999999


def repair_schedules_in_data(data: list) -> int:
    """
    Set scheduled_start_time[k] = end_time[k-1] (departure time toward activity k).
    In the cyclic model: scheduled_start_time[0] = end_time[last] % 86400.
    Returns the number of values written.
    """
    repaired = 0
    for entry in data:
        identity   = entry.get("identity", {})
        activities = identity.get("activities", [])
        acts = [a for a in activities if a.get("start_time") is not None]
        n = len(acts)

        if n == 0:
            continue

        if n == 1:
            target = acts[0].get("start_time", 0.0)
            if acts[0].get("scheduled_start_time") != target:
                acts[0]["scheduled_start_time"] = target
                repaired += 1
            continue

        for i in range(n):
            prev = acts[(i - 1) % n]
            curr = acts[i]
            departure_time = prev.get("end_time", 0.0) % 86400
            if curr.get("scheduled_start_time") != departure_time:
                curr["scheduled_start_time"] = departure_time
                repaired += 1

    return repaired


total_repaired = 0
for fname, data in all_data.items():
    n = repair_schedules_in_data(data)
    total_repaired += n
    print(f"{fname}: {n} scheduled_start_time set")

print(f"\nTotal values written across all files: {total_repaired}")

toulouse_population_100.json: 407 scheduled_start_time set

Total values written across all files: 407


## Section 5b bis — Schedule Gap Repair

Adjust `end_time` / `start_time` so every consecutive pair satisfies:
```
gap = start_time[k] - end_time[k-1]  >=  min_trip + 5 min
```
- **Case A** (flex predecessor): compress `end_time` backward chain up to `D_MIN=5 min`.
- **Case B** (rigid predecessor, flex successor): shift `start_time` to `prev.end + trip + 5 min`.
- **Case C** (rigid → rigid): depart ASAP, lateness accepted.

Idempotent — safe to re-run. Matches the logic in `handle/application.py`.

In [ ]:
from typing import List, Optional


MIN_ACTIVITY_DELAY = 15 * 60   # 15 min in seconds
_SCHED_MARGIN      = 10 * 60   # 10 min margin in seconds
_RIGID_SCH         = ['education', 'work','leisure','other','shop','home']


def _pick_trip_s(tf: dict) -> int:
    """Fastest available mode travel time (car > bicycle > foot)."""
    for mode in ('car', 'bicycle', 'foot'):
        v = tf.get(mode)
        if v and v.get('duration_s', 0) > 0:
            return int(v['duration_s'])
    return 0


def get_priority(purpose):
    if purpose not in _RIGID_SCH:
        print(f"  !! Unknown purpose '{purpose}' not in priority list {_RIGID_SCH}")
        return len(_RIGID_SCH)
    return _RIGID_SCH.index(purpose)

def describe_activities(activities: list) -> str:
    """Return a one-line chronological description of a person's activity sequence.
    Accepts Activity model objects or plain dicts.

    Example: [work: sched=07h40, start=08h20, end=12h00] -> [leisure: sched=12h00, start=12h15, end=13h30]
    """
    def _fmt(seconds) -> str:
        if seconds is None:
            return "?"
        total = int(seconds) % (24 * 3600)
        h, rem = divmod(total, 3600)
        m = rem // 60
        return f"{h:02d}h{m:02d}"

    def _get(act, key):
        return act[key] if isinstance(act, dict) else getattr(act, key)

    parts = [
        f"[{_get(a, 'purpose')}: sched={_fmt(_get(a, 'scheduled_start_time'))}, start={_fmt(_get(a, 'start_time'))}, end={_fmt(_get(a, 'end_time'))}]"
        for a in activities
    ]
    return " -> ".join(parts)

def shift_time(times, decalage):
    return [(t + decalage) % 86400 if t is not None else None for t in times]


def compress_chain_backward(person_id, activities, act_index, travel_s):
    act      = activities[act_index]
    prev = activities[act_index - 1] if act_index > 0 else activities[-1]
    next_act = activities[act_index + 1] if act_index < len(activities) - 1 else activities[0]

    if (act is None or prev is None or next_act is None):
        print(f"  !! Person {person_id}: Missing activity for compression at index {act_index} (prev={prev}, act={act}, next={next_act})")
        return False

    scheduled_start_time = (act['start_time'] - travel_s - _SCHED_MARGIN) % 86400
    current_delay     = (prev['end_time'] - scheduled_start_time + 86400) % 86400
    previous_act_marging = (prev['end_time'] - prev['start_time'] - _SCHED_MARGIN - MIN_ACTIVITY_DELAY- travel_s+ 86400) % 86400
    current_act_marging = (act['end_time'] - act['start_time'] - _SCHED_MARGIN - MIN_ACTIVITY_DELAY- travel_s+ 86400) % 86400
    next_act_marging = (next_act['end_time'] - next_act['start_time'] - _SCHED_MARGIN - MIN_ACTIVITY_DELAY- travel_s+ 86400) % 86400

    # else: 
    #     print(f"Person {person_id}: Trying to compress activity at index {act_index} with travel_s={travel_s} seconds, decalage={decalage} seconds")

    if current_delay <= 0:
        act['scheduled_start_time'] = scheduled_start_time
        return False
    
    # Cas 1: Réduire la durée de l'activité précédente si elle est moins prioritaire
    if (get_priority(act['purpose']) < get_priority(prev['purpose'])) and \
        previous_act_marging > current_delay:
        prev['end_time'] = shift_time([prev['end_time']], -current_delay)[0]
        act['scheduled_start_time'] = scheduled_start_time
        return False

    # Cas 2: Réduire la durée de l'activité suivante si elle est moins prioritaire
    if next_act and (get_priority(act['purpose']) < get_priority(next_act['purpose'])) and \
        next_act_marging > current_delay:

        act['scheduled_start_time'], act['start_time'], act['end_time'] = shift_time([act['scheduled_start_time'], act['start_time'], act['end_time']], current_delay)
        next_act['scheduled_start_time'], next_act['start_time'] = shift_time([next_act['scheduled_start_time'], next_act['start_time']], current_delay)
        return True
    
    # Cas 3: Réduire la durée de l'activité courante si elle est moins prioritaire que la précédente et la suivante
    if current_act_marging > current_delay:
        act['scheduled_start_time'], act['start_time'] = shift_time([act['scheduled_start_time'], act['start_time']], current_delay)
        return True
        
    # Cas 4: Réduire la durée de l'activité précédente (sans condition de priorité, si elle est suffisamment longue)
    if previous_act_marging > current_delay:
        prev['end_time'] = shift_time([prev['end_time']], -current_delay)[0]
        return False

    # Cas 5: Réduire la durée de l'activité suivante (sans condition de priorité, si elle est suffisamment longue)
    if next_act_marging > current_delay:
        act['scheduled_start_time'], act['start_time'], act['end_time'] = shift_time([act['scheduled_start_time'], act['start_time'], act['end_time']], current_delay)
        next_act['scheduled_start_time'], next_act['start_time'] = shift_time([next_act['scheduled_start_time'], next_act['start_time']], current_delay)
        return True

    # Cas 6: Décaler l'activité précédente si elle est moins prioritaire
    if get_priority(act['purpose']) < get_priority(prev['purpose']):
        prev['scheduled_start_time'],prev['start_time'],prev['end_time']  = shift_time([prev['scheduled_start_time'],prev['start_time'],prev['end_time']], -current_delay)
        if is_verbose: print("6")
        return True

    # Cas 7: Décaler l'activité courante (et la suivante pour préserver l'ordre)
    act['scheduled_start_time'],act['start_time'],act['end_time'] = shift_time([act['scheduled_start_time'], act['start_time'], act['end_time']], current_delay)
    next_act['scheduled_start_time'],next_act['start_time'],next_act['end_time'] = shift_time([next_act['scheduled_start_time'], next_act['start_time'], next_act['end_time']], current_delay)
    return True




def adjust_schedules_for_travel(data: list) -> int:
    """
    Repair scheduled_start_time / end_time / start_time so that every
    consecutive pair satisfies:
        gap = start_time[k] - end_time[k-1]  >=  min_trip + _SCHED_MARGIN
    Uses priority-based compression via compress_chain_backward.
    Returns number of activities processed.
    """
    MAX_ITER = 20
    n_adj = 0
    for _iter in range(MAX_ITER):
        print(f"Adjusting schedules for travel (iteration {_iter + 1}/{MAX_ITER})…")
        for entry in data:
            acts = entry.get('identity', {}).get('activities', [])
            pid        = entry.get("person_id", "?")
            if pid == "500928":
                print(f"\nPerson {pid} activities before iteration {_iter + 1}: {describe_activities(acts)}")   
            if len(acts) >= 2:
                for i in range(0, len(acts)):
                    act      = acts[i]
                    travel_s = _pick_trip_s(act.get('transfert_from_previous_location') or {})
                    if not travel_s:
                        continue
                    if compress_chain_backward(pid,acts, i, travel_s):
                        n_adj += 1
        # if not moved:
        #     break
    return n_adj


total_adj = 0
for fname, data in all_data.items():
    print(f"\nProcessing schedule adjustments for {fname}…")
    n = adjust_schedules_for_travel(data)
    total_adj += n
    print(f"{fname}: {n} activities schedule-adjusted")
print(f"\nTotal adjusted: {total_adj}")



Processing schedule adjustments for toulouse_population_100.json…
Adjusting schedules for travel (iteration 1/20)…
NEEDED = 4386.0
NEEDED = 4509.0
NEEDED = 4386.0
NEEDED = 4509.0
NEEDED = 373.0
NEEDED = 373.0
NEEDED = 692.0
NEEDED = 727.0
NEEDED = 86035.0
NEEDED = 1705.0
NEEDED = 84768.0
NEEDED = 84468.0
NEEDED = 713.0
NEEDED = 709.0
NEEDED = 1367.0
NEEDED = 1589.0
NEEDED = 1846.0
NEEDED = 1546.0
NEEDED = 1546.0
NEEDED = 1846.0
NEEDED = 346.0
NEEDED = 346.0
NEEDED = 2300.0
NEEDED = 2382.0
NEEDED = 940.0
NEEDED = 85512.0
NEEDED = 2323.0
NEEDED = 2280.0
NEEDED = 2323.0
NEEDED = 2280.0
NEEDED = 86039.0
NEEDED = 1395.0
NEEDED = 901.0
NEEDED = 1262.0
NEEDED = 1565.0
NEEDED = 684.0
NEEDED = 1609.0
NEEDED = 269.0
NEEDED = 2049.0
NEEDED = 2417.0
NEEDED = 601.0
NEEDED = 1824.0
NEEDED = 1824.0
NEEDED = 84601.0
NEEDED = 813.0
NEEDED = 788.0
NEEDED = 85690.0
NEEDED = 83245.0
NEEDED = 82511.0
NEEDED = 82871.0
NEEDED = 81.0
NEEDED = 86164.0
NEEDED = 792.0
NEEDED = 4731.0
NEEDED = 5094.0
NEEDED = 60

## Section 5c — Travel Gap Feasibility Check

For each consecutive pair `(act[k] → act[k+1])`, verify that the ENTD gap
`start_time[k+1] - end_time[k]` is physically feasible:

```
gap  >=  min_travel_time(foot, bicycle[, car])  +  margin (5 min)
```

Data is **not modified** — only warnings are emitted.

In [25]:
VALIDATION_MARGIN_S = 5 * 60  # 5-minute buffer


def validate_travel_gaps(data: list, margin_s: int = VALIDATION_MARGIN_S) -> list[dict]:
    """
    For each consecutive activity pair (including cyclic last→first), check:
        gap = start_time[k+1] - end_time[k]  >=  min_travel_time + margin_s
    Returns a list of warning dicts for infeasible pairs.
    """
    warnings = []
    for entry in data:
        identity   = entry.get("identity", {})
        activities = identity.get("activities", [])
        traits     = identity.get("traits_json", {})
        has_car    = traits.get("number_of_cars", 0) > 0
        pid        = entry.get("person_id", "?")

        for prev_i, curr_i, _ in _activity_index_pairs(activities, has_car):
            prev_act = activities[prev_i]
            curr_act = activities[curr_i]

            prev_et = prev_act.get("end_time")
            curr_st = curr_act.get("start_time")
            if prev_et is None or curr_st is None:
                continue

            gap = (curr_st + 86400 - prev_et) % 86400

            min_trip = _min_trip_s(curr_act)

            if gap < min_trip + margin_s:
                warnings.append({
                    "person_id":   pid,
                    "from_idx":    prev_i,
                    "to_idx":      curr_i,
                    "from_purpose": prev_act.get("purpose"),
                    "to_purpose":  curr_act.get("purpose"),
                    "previous_end": pd.to_datetime(prev_et, unit="s").strftime('%H:%M'),
                    "current_start": pd.to_datetime(curr_st, unit="s").strftime('%H:%M'),
                    "gap_m":       round(gap/60, 1),
                    "min_trip_m":  round(min_trip/60, 1),
                    "deficit_m":   round((min_trip + margin_s - gap)/60, 1),
                })
    return warnings


all_warnings = []
for fname, data in all_data.items():
    w = validate_travel_gaps(data)
    all_warnings.extend([{**wi, "file": fname} for wi in w])
    print(f"{fname}: {len(w)} infeasible gap(s)")

print(f"\nTotal infeasible pairs across all files: {len(all_warnings)}")
pd.set_option('display.max_rows', None) 
if all_warnings:
    df_warn = pd.DataFrame(all_warnings)
    print(f"\nTop 100 worst deficits (gap too short by X seconds):")
    display(df_warn.nlargest(100, "deficit_m")[
        ["file", "person_id", "from_purpose", "to_purpose", "previous_end", "current_start", "gap_m", "min_trip_m", "deficit_m"]
    ])

toulouse_population_100.json: 0 infeasible gap(s)

Total infeasible pairs across all files: 0


## Section 6 — Vérification finale & Sauvegarde atomique

Réalise un dernier passage de `check_enrichment` sur chaque fichier pour confirmer l'absence
de routes manquantes et de flags PT non renseignés, puis sauvegarde chaque fichier de manière
atomique (écriture dans `.json.tmp` + renommage) pour éviter tout fichier corrompu en cas
d'interruption.

In [ ]:
all_ok = True
for fname, data in all_data.items():
    stats = check_enrichment(data)
    still_missing = stats["missing_routes"] + stats["missing_any"]
    pt_missing    = stats["missing_pt"]
    status = "OK" if still_missing == 0 and pt_missing == 0 else "INCOMPLETE"
    if status != "OK":
        all_ok = False
    print(f"{fname}: {status}  (missing_routes={stats['missing_routes']}, "
          f"missing_any_mode={stats['missing_any']}, missing_pt={pt_missing})")

# Sauvegarde atomique : écriture dans un fichier .tmp puis renommage,
# pour éviter tout fichier corrompu en cas d'interruption en cours d'écriture
saved = []
for fname, data in all_data.items():
    dest     = POP_DIR / fname
    tmp_path = dest.with_suffix(".json.tmp")
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.rename(tmp_path, dest)
    saved.append(fname)

print(f"\nSaved {len(saved)} file(s): {saved}")
if all_ok:
    print("All files are correctly completed.")
else:
    print("WARNING: some entries still incomplete (unreachable routes or missing coordinates).")